# Titanic — Survival Prediction with **Linear Regression only**

**Rule of the task:** the model is fixed — we may only use plain `LinearRegression`.
Linear regression is a deliberately awkward tool here: it predicts a *number*, but the
answer is *yes / no*. The challenge is therefore **not** in picking a model — it is in
**what we feed it** (features) and **where we cut** the continuous output into 0/1.

We are not chasing a perfect leaderboard score (those are people who looked up the public
answers). We chase an *honest* result we understand.

### Levers we pull (all on the data side)
1. **Fill missing `Age` meaningfully** — median per *Title* group, not one global mean.
2. **Encode categories as numbers** — `Sex`, `Pclass`, `Embarked`.
3. **Build new features** — `FamilySize = SibSp+Parch+1`, `IsAlone`, `Title` from the name
   (Mr / Mrs / Miss / Master / Rare).
4. **`Family_Survival`** — the strongest lever: did a passenger's *known* relatives (same
   surname+fare, or same ticket) live or die? Families tended to share a fate.
5. **Bin `Age` and `Fare`** so a straight line can use them robustly.
6. **Tune the 0/1 threshold** on *out-of-fold* predictions, so the cut is honest.

> A leaner, well-chosen feature set generalises **better** than a big one — the first
> version of this notebook scored 0.835 in cross-validation but only ~0.78 on Kaggle
> (overfitting). Here we trade some CV score for a model that travels better to unseen data.

In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import accuracy_score

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
print('train:', train.shape, '  test:', test.shape)
print('\nMissing values per column (train):')
print(train.isna().sum()[train.isna().sum() > 0])
train.head()

train: (891, 12)   test: (418, 11)

Missing values per column (train):
Age         177
Cabin       687
Embarked      2
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1. Feature engineering

We transform `train` and `test` **together** (one combined frame). The transforms are
deterministic and never touch a passenger's *own* `Survived`, so there is no target leakage.

The headline feature is **`Family_Survival`**: people travelled in groups, and whole
families tended to live or die together. For each passenger we look at the *known* outcomes
of their relatives (same surname **and** fare, or the same ticket) — **excluding the
passenger themselves** — and record whether the group lived (1), died (0), or is unknown
(0.5). For a *test* passenger this is pure signal: we only ever read labels from the
training rows, never their own (which doesn't exist). This is the single strongest honest
lever on this dataset.

We also **bin `Age` and `Fare`** into ordered buckets. Linear regression fits a straight
line; raw, skewed `Fare` and the non-monotonic effect of `Age` (children survive more, then
it dips) are captured more robustly by a handful of bins than by the raw number.

In [8]:
n_train = len(train)
combined = pd.concat([train, test], sort=False, ignore_index=True)

# --- Title extracted from the Name: 'Braund, Mr. Owen Harris' -> 'Mr' ---
combined['RawTitle'] = combined['Name'].str.extract(r',\s*([^.]+)\.')[0].str.strip()
title_map = {
    'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master',
    'Ms': 'Miss', 'Mlle': 'Miss', 'Mme': 'Mrs', 'Lady': 'Rare', 'Countess': 'Rare',
    'Dona': 'Rare', 'Don': 'Rare', 'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dr': 'Rare',
    'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare', 'Capt': 'Rare',
}
combined['Title'] = combined['RawTitle'].map(title_map).fillna('Rare')

# --- Encode Sex; family-size features ---
combined['Sex_female'] = (combined['Sex'] == 'female').astype(int)
combined['FamilySize'] = combined['SibSp'] + combined['Parch'] + 1
combined['IsAlone']    = (combined['FamilySize'] == 1).astype(int)

# --- Fill missing Age by Title-group median, Fare by Pclass median, Embarked by mode ---
combined['Age']  = combined['Age'].fillna(combined.groupby('Title')['Age'].transform('median'))
combined['Fare'] = combined['Fare'].fillna(combined.groupby('Pclass')['Fare'].transform('median'))
combined['Embarked'] = combined['Embarked'].fillna(combined['Embarked'].mode()[0])

# === KEY FEATURE: Family_Survival ==========================================
# Did this passenger's KNOWN relatives survive? We read only OTHER members'
# labels (drop self), so a test passenger's own (absent) label is never used.
combined['LastName'] = combined['Name'].str.split(',').str[0]
combined['Family_Survival'] = 0.5  # default: no information

# (a) group by surname + fare (same family usually shares both)
for _, grp in combined.groupby(['LastName', 'Fare']):
    if len(grp) > 1:
        for idx in grp.index:
            others = grp.drop(idx)['Survived']
            if others.max() == 1.0:
                combined.loc[idx, 'Family_Survival'] = 1.0
            elif others.min() == 0.0:
                combined.loc[idx, 'Family_Survival'] = 0.0

# (b) fall back to ticket groups for anyone still unknown / marked dead
for _, grp in combined.groupby('Ticket'):
    if len(grp) > 1:
        for idx in grp.index:
            if combined.loc[idx, 'Family_Survival'] in (0.0, 0.5):
                others = grp.drop(idx)['Survived']
                if others.max() == 1.0:
                    combined.loc[idx, 'Family_Survival'] = 1.0
                elif others.min() == 0.0:
                    combined.loc[idx, 'Family_Survival'] = 0.0

print('Family_Survival distribution (0=group died, 0.5=unknown, 1=group lived):')
print(combined['Family_Survival'].value_counts())
# ===========================================================================

# --- Bin Age and Fare into ordered buckets (robust for a linear model) ---
combined['AgeBin']  = pd.cut(combined['Age'], bins=[0, 14, 25, 35, 45, 60, 200], labels=False)
combined['FareBin'] = pd.qcut(combined['Fare'], 5, labels=False)

# --- One-hot for Title (keeps 'Master' = young boys, not captured by Sex) ---
combined = pd.get_dummies(combined, columns=['Title'], dtype=int)

combined[['Sex_female', 'Pclass', 'AgeBin', 'FareBin', 'FamilySize',
          'Family_Survival', 'Title_Master']].head()

Family_Survival distribution (0=group died, 0.5=unknown, 1=group lived):
Family_Survival
0.5    763
1.0    303
0.0    243
Name: count, dtype: int64


,Sex_female,Pclass,AgeBin,FareBin,FamilySize,Family_Survival,Title_Master
0,0,3,1,0,2,0.5,0
1,1,1,3,4,2,0.5,0
2,1,3,2,1,1,0.5,0
3,1,1,2,4,2,0.0,0
4,0,3,2,1,1,0.5,0


In [9]:
# A LEAN feature set: every column carries distinct signal -> generalises better.
feature_cols = [
    'Family_Survival',   # known fate of relatives  (strongest)
    'Sex_female',        # women survived far more
    'Pclass',            # 1st > 2nd > 3rd  (ordinal, kept numeric)
    'AgeBin',            # children favoured
    'FareBin',           # wealth proxy
    'FamilySize',        # mid-size families did best
    'Title_Master',      # young boys (signal beyond Sex)
]

X      = combined.loc[:n_train - 1, feature_cols].astype(float)
y      = train['Survived'].astype(float)
X_test = combined.loc[n_train:, feature_cols].astype(float).reset_index(drop=True)

print('X:', X.shape, '  X_test:', X_test.shape)

X: (891, 7)   X_test: (418, 7)


## 2. Fit linear regression & tune the 0/1 threshold

Linear regression outputs a continuous score; we need a cut-off to get a yes/no label. We
use **5-fold cross-validation** to pick the threshold that maximises out-of-fold accuracy.

> ⚠️ Honesty note: `Family_Survival` is built from *all* training labels, so during CV a
> validation passenger's feature was partly informed by training rows in other folds. That
> makes the CV number **optimistic** — it is *not* a clean estimate anymore. We keep CV only
> to choose a sensible threshold; the true verdict is the **Kaggle public score**, where the
> test passengers' own labels were never available to the feature.

In [10]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
lin = LinearRegression()

# Out-of-fold continuous predictions (each row predicted by a model that never saw it)
oof_scores = cross_val_predict(lin, X, y, cv=cv)

# Sweep the threshold and keep the one with the best out-of-fold accuracy
thresholds = np.arange(0.30, 0.71, 0.01)
accuracies = [accuracy_score(y, (oof_scores >= t).astype(int)) for t in thresholds]
best_t   = thresholds[int(np.argmax(accuracies))]
best_acc = max(accuracies)

print(f'Best threshold      : {best_t:.2f}')
print(f'Out-of-fold accuracy: {best_acc:.4f}')
print(f'(accuracy at naive 0.50 cut: {accuracy_score(y, (oof_scores >= 0.5).astype(int)):.4f})')

Best threshold      : 0.54
Out-of-fold accuracy: 0.8530
(accuracy at naive 0.50 cut: 0.8474)


In [11]:
# Which features push survival up / down? (linear coefficients are directly readable)
lin.fit(X, y)
coefs = pd.Series(lin.coef_, index=feature_cols).sort_values()
print('Most negative (lowers survival) -> most positive (raises survival):')
print(coefs)

Most negative (lowers survival) -> most positive (raises survival):
Pclass            -0.100028
FamilySize        -0.053172
AgeBin            -0.024013
FareBin            0.021216
Family_Survival    0.360589
Title_Master       0.536559
Sex_female         0.563210
dtype: float64


## 3. Train on all data, predict the test set, write the submission

In [12]:
# Final model: fit on the FULL training set, apply the threshold chosen by CV
final = LinearRegression().fit(X, y)
test_scores = final.predict(X_test)
predictions = (test_scores >= best_t).astype(int)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions,
})
submission.to_csv('submission.csv', index=False)

print('Wrote submission.csv  ', submission.shape)
print('Predicted survivors:', int(predictions.sum()), 'of', len(predictions),
      f'({predictions.mean():.1%})')
submission.head()

Wrote submission.csv   (418, 2)
Predicted survivors: 149 of 418 (35.6%)


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


## Summary

- **Model:** still fixed to plain `LinearRegression` — no other estimator, no regularisation.
- **What changed from v1 (which scored 0.77751 on Kaggle):**
  - Added **`Family_Survival`** — the known fate of each passenger's relatives. The
    coefficients show it is the third-strongest driver (+0.36), right behind `Sex_female`
    (+0.56) and `Title_Master` (+0.54, young boys).
  - **Slimmed 19 features down to 7**, dropping overlapping/noisy columns (`Fare`,
    `FarePerPerson`, `SibSp`, `Parch`, the `Embarked` and extra `Title` dummies). Fewer
    features ⇒ less overfitting ⇒ the model travels better to the unseen test set.
  - **Binned `Age` and `Fare`** so the straight-line model handles their non-linear effects.
- **Honesty caveat:** the cross-validation number (~0.85) is now *optimistic*, because
  `Family_Survival` peeks at training labels across folds. It is fine for picking the
  threshold, but the real measure is the **Kaggle public score**.
- **Expectation:** this should land around **0.78–0.79** on Kaggle — an honest improvement
  over 0.77751, still fully understood, and still within the linear-regression rule.

`submission.csv` is regenerated and ready to upload.